In [ ]:
from dataset.documents import load_vietnamese_legal_documents,chunk_documents,load_vietnamese_legal_datasets
from dataset.vectorstore import VectorStoreDB, VectorStoreType
from rag.llm.embeddings import embedding_factory, EmbeddingProvider
from dataset.sql import SQLiteDatabase, VietnamLawModel

from settings.settings import (
    EMBEDDING_MODEL,
    SENTENCE_TRANFORMER_MODEL,
    LLM_MODEL,
    PERSIST_DIR,
    CHUNK_SIZE,
    CHUNK_OVERLAPPED,
 )

In [ ]:
docs = load_vietnamese_legal_documents(start=15000,limit=5000) #15000
datasets = load_vietnamese_legal_datasets(start=15000,limit=5000) #15000
chunked_doc = chunk_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAPPED)
embedder = embedding_factory(EmbeddingProvider.HUGGINGFACE, EMBEDDING_MODEL)
db = VectorStoreDB(
    type=VectorStoreType.QDRANT,
    name="vietnamese_legal_docs",
    embedder=embedder
)
sql = SQLiteDatabase(name="vietnam_laws.db", path = "../db/sql", model=VietnamLawModel)

In [ ]:
docs[0]

In [ ]:
sql.add_many(datasets)

In [ ]:
print(f"data = {sql.get_at_index(0)}")

In [ ]:
db.build()

In [ ]:
db.add(
    documents=chunked_doc,
    batch_size=16,
    wait=False,
    timeout=120,
)